# Demo 5: Incident Narrative & Risk Story

## The Real-World Problem

**Scenario:** An incident happened. Here's the raw timeline:
```
09:02 - Spike in login failures
09:17 - Support confirms 12 users affected
09:41 - Engineering finds root cause (OAuth config)
10:05 - Finance flags ARR risk if not fixed today
10:45 - Patch deployed to staging
11:15 - Fix verified in production
12:00 - Customer notifications sent
```

**The Challenge:**
- ❌ Without Ninai: Executive reads timestamps and fragments, has to piece together the story themselves
  - "What actually went wrong?"
  - "How bad was it really?"
  - "What do we do next?"
- ✅ With Ninai: One `explain()` call generates a coherent narrative
  - "Here's what happened, why it mattered, and what we did"
  - Risk score: embedded in the story
  - 24-hour action plan: automatically generated

**What Ninai Does (using Cognitive Gateway):**
1. **Reads the timeline** — Ingests fragmented incident events
2. **Extracts causality** — Links impact to root cause
3. **Scores risk** — Quantifies severity based on business impact
4. **Generates narrative** — Transforms raw events into a coherent story
5. **Recommends actions** — 24h follow-up plan with priorities
6. **Produces executive summary** — One-pager ready for leadership

**From timeline fragments to polished executive briefing.** ⏱️📊

---

## Learning Goals

1. **Understand the problem** — Raw timelines are hard to interpret
2. **Create incident timeline** — Multiple discrete events with different facets
3. **Use Cognitive Gateway** — Delegate narrative synthesis to backend agents
4. **Get executive summary** — Coherent story with risk scoring
5. **See causality linking** — How Ninai connects events into understanding

## What You'll Learn

- ✓ Store timeline events as individual memories
- ✓ Call `client.cognitive.gateway.explain()` to synthesize narrative
- ✓ Receive risk-scored incident story
- ✓ See 24-hour action plan with priorities
- ✓ Understand how Ninai turns fragments into insight

**Raw timeline → Executive narrative with risk assessment.** 📈

In [1]:
## Step 1: Setup and Login


from ninai import NinaiClient
import uuid

BASE_URL = 'https://admin.ninai.sansten.com/api/v1'
EMAIL = 'demo@ninai.dev'
PASSWORD = 'demo1234'
ORG_SLUG = 'default'

client = NinaiClient(base_url=BASE_URL)
client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)
seed = str(uuid.uuid4())[:8]

print(f"OK: Authenticated")
print(f"Seed: {seed}")
print(f"Ready for Step 2")

OK: Authenticated
Seed: 42625412
Ready for Step 2


In [6]:
## Step 2: Create Incident Timeline

#**Important:** Do NOT re-run Step 1 after this point. The seed will change.

#Create timeline events as separate memories. Each event captures a different facet of the incident's evolution.

import time
from datetime import datetime, timezone
from ninai.exceptions import ServerError, AuthenticationError

INCIDENT_DATE = '2026-04-15'

def to_utc_iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat().replace('+00:00', 'Z')

def create_memory_with_retry(content, tags, metadata=None, occurred_at=None, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return client.memories.create(
                content=content,
                source_type='manual',
                tags=tags,
                metadata=metadata or {},
                occurred_at=occurred_at
            )
        except AuthenticationError:
            # Refresh session if token expired during longer notebook runs.
            client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)
        except ServerError:
            if attempt < attempts:
                time.sleep(0.6 * attempt)
    return None

print("=" * 80)
print("STEP 2: Creating Incident Timeline")
print("=" * 80)
print(f"\nSeed: {seed}")
print("DO NOT re-run Step 1 or the seed will change!\n")

# Create timeline events with explicit datetime metadata for time-series reasoning
timeline_events = [
    {
        'time': '09:02',
        'phase': 'DETECTION',
        'details': 'Spike in enterprise login failures detected. Error rate jumped from 0.1% to 8.5% in 30 seconds.'
    },
    {
        'time': '09:17',
        'phase': 'CUSTOMER IMPACT',
        'details': 'Support confirms 127 users affected. Top account: ACME (5-year customer, $50K MRR). Reset workflow failing for all enterprise customers.'
    },
    {
        'time': '09:41',
        'phase': 'ROOT CAUSE',
        'details': 'Engineering identifies OAuth2 audience mismatch in production JWT validation. Mismatch introduced 2 weeks ago via config drift (prod !== staging).'
    },
    {
        'time': '10:05',
        'phase': 'BUSINESS RISK',
        'details': 'Finance marks incident as ARR risk. Potential customer churn: $120K+ if unresolved by EOD. SLA breach: 99.95% -> 97.8%.'
    },
    {
        'time': '10:45',
        'phase': 'FIX DEPLOYED',
        'details': 'Engineering patches JWT_AUDIENCE env var in production canary. E2E tests pass. No database changes required.'
    },
    {
        'time': '11:15',
        'phase': 'VERIFICATION',
        'details': 'Production fix verified. Error rate back to 0.1%. 50 additional users tested successfully.'
    },
    {
        'time': '12:00',
        'phase': 'COMMUNICATION',
        'details': 'Support sends customer briefing to ACME + affected accounts. No customer churn detected. Executive note scheduled for 16:00.'
    },
    {
        'time': '12:30',
        'phase': 'POST-INCIDENT',
        'details': 'Engineering begins root cause analysis. Action items: config drift detection, automated staging-prod validation, JWT audience test suite.'
    },
]

created_ids = []
failed_writes = 0
for idx, event in enumerate(timeline_events, 1):
    event_dt = datetime.fromisoformat(f"{INCIDENT_DATE}T{event['time']}:00").replace(tzinfo=timezone.utc)
    event_time_iso = to_utc_iso_z(event_dt)
    content = f"{event['phase']}: {event['details']}"
    metadata = {
        'event_time': event_time_iso,
        'event_date': INCIDENT_DATE,
        'event_clock': event['time'],
        'phase': event['phase'],
        'seed': seed,
        'demo': 'demo_5_executive_incident_story'
    }

    mem = create_memory_with_retry(
        content=content,
        tags=['incident', 'timeline', 'p1', seed],
        metadata=metadata,
        occurred_at=event_dt,
        attempts=3
    )

    label = f"{event['time']} - {event['phase']}"
    if mem is not None:
        created_ids.append(mem.id)
        print(f"Created memory {idx}/{len(timeline_events)}: {label}")
    else:
        failed_writes += 1
        print(f"Write skipped after retries {idx}/{len(timeline_events)}: {label}")

# Build structured timeline payload for Step 3 and consolidated narrative memory
timeline_rows = [
    {
        'event_time': f"{INCIDENT_DATE}T{e['time']}:00Z",
        'phase': e['phase'],
        'details': e['details']
    }
    for e in timeline_events
]
timeline_rows.sort(key=lambda x: x['event_time'])

timeline_text = "\n".join([
    f"{row['event_time']} | {row['phase']} | {row['details']}"
    for row in timeline_rows
] )
timeline_memory_id = None
summary_mem = create_memory_with_retry(
    content=f"INCIDENT TIMELINE SUMMARY\n{timeline_text}",
    tags=['incident', 'timeline', 'summary', seed],
    metadata={
        'timeline_start': timeline_rows[0]['event_time'],
        'timeline_end': timeline_rows[-1]['event_time'],
        'timeline_event_count': len(timeline_rows),
        'seed': seed,
        'demo': 'demo_5_executive_incident_story'
    },
    occurred_at=datetime.fromisoformat(timeline_rows[-1]['event_time'].replace('Z', '+00:00')),
    attempts=3
)
if summary_mem is not None:
    timeline_memory_id = summary_mem.id

print(f"\n✓ Total events prepared: {len(timeline_events)}")
print(f"✓ Memory writes succeeded: {len(created_ids)}")
print(f"✓ Memory writes failed: {failed_writes + (0 if timeline_memory_id else 1)}")
print(f"✓ Tagged with seed: {seed}")
print(f"✓ Datetime passed as occurred_at in UTC and mirrored in metadata.event_time")
if timeline_memory_id:
    print(f"✓ Timeline summary memory id: {timeline_memory_id}")
else:
    print("Timeline summary memory unavailable; Step 3 will use direct synthesis fallback")
print(f"✓ Timeline span: {timeline_rows[0]['event_time']} -> {timeline_rows[-1]['event_time']}")
print(f"\nProceed to Step 3")

STEP 2: Creating Incident Timeline

Seed: 42625412
DO NOT re-run Step 1 or the seed will change!

Write skipped after retries 1/8: 09:02 - DETECTION
Write skipped after retries 2/8: 09:17 - CUSTOMER IMPACT
Write skipped after retries 3/8: 09:41 - ROOT CAUSE
Write skipped after retries 4/8: 10:05 - BUSINESS RISK
Write skipped after retries 5/8: 10:45 - FIX DEPLOYED
Write skipped after retries 6/8: 11:15 - VERIFICATION
Write skipped after retries 7/8: 12:00 - COMMUNICATION
Write skipped after retries 8/8: 12:30 - POST-INCIDENT

✓ Total events prepared: 8
✓ Memory writes succeeded: 0
✓ Memory writes failed: 9
✓ Tagged with seed: 42625412
✓ Datetime stored separately in metadata field: event_time
Timeline summary memory unavailable; Step 3 will use direct synthesis fallback
✓ Timeline span: 2026-04-15T09:02:00Z -> 2026-04-15T12:30:00Z

Proceed to Step 3


## Step 3: Synthesize Executive Narrative

Run the next cell to invoke Cognitive Gateway to transform timeline into narrative.

Ninai will:
1. Read all timeline events
2. Link causality (detection → impact → root cause → fix → resolution)
3. Extract key metrics and risk assessment
4. Generate coherent narrative with risk score
5. Produce 24-hour action plan

Expected outcome:
- A polished executive briefing
- Risk score (0-100) based on severity
- Narrative connecting events into understanding
- 24-hour follow-up actions
- Key metrics embedded in story

In [5]:
# Step 3: Synthesize Executive Narrative via Cognitive Gateway
print("=" * 80)
print("STEP 3: Executive Narrative Synthesis")
print("=" * 80)
print(f"\nAnalyzing incident timeline with seed: {seed}\n")

# Prefer structured in-kernel timeline data from Step 2
if 'timeline_rows' in globals() and timeline_rows:
    source_rows = timeline_rows
    print(f"Using structured in-session timeline data: {len(source_rows)} events")
else:
    source_rows = []
    if 'timeline_events' in globals() and timeline_events:
        # Backward-compatible conversion for older cell state (string or dict entries)
        for idx, item in enumerate(timeline_events, 1):
            if isinstance(item, dict) and {'time', 'phase', 'details'}.issubset(item.keys()):
                source_rows.append({
                    'event_time': f"{INCIDENT_DATE}T{item['time']}:00Z" if 'INCIDENT_DATE' in globals() else item['time'],
                    'phase': item['phase'],
                    'details': item['details']
                })
            else:
                source_rows.append({
                    'event_time': f"event_{idx}",
                    'phase': 'TIMELINE_EVENT',
                    'details': str(item)
                })

if not source_rows:
    print("\nERROR: No timeline events found.")
    print("Check Step 2 or ensure you did NOT re-run Step 1")
else:
    source_rows.sort(key=lambda x: x['event_time'])
    timeline_text = "\n".join([
        f"{row['event_time']} | {row['phase']} | {row['details']}"
        for row in source_rows
    ])
    
    try:
        print("\n" + "=" * 80)
        if 'timeline_memory_id' in globals() and timeline_memory_id:
            print("INVOKING COGNITIVE GATEWAY EXPLAIN (memory-backed)")
            print("=" * 80)
            print(f"\nCalling client.cognitive.gateway.explain(memory_id={timeline_memory_id})...")
            narrative_result = client.cognitive.gateway.explain(memory_id=timeline_memory_id)
        else:
            print("INVOKING COGNITIVE GATEWAY PLAN (direct synthesis fallback)")
            print("=" * 80)
            print("\nCalling client.cognitive.gateway.plan() with structured timeline context...")
            narrative_result = client.cognitive.gateway.plan(
                goal="Generate executive incident narrative with risk score and 24h action plan",
                context={
                    'incident_timeline': timeline_text,
                    'incident_timeline_events': source_rows,
                    'timeline_start': source_rows[0]['event_time'],
                    'timeline_end': source_rows[-1]['event_time'],
                    'output_format': 'executive_brief',
                    'required_fields': [
                        'risk_score',
                        'risk_level',
                        'narrative',
                        'causality_chain',
                        'key_metrics',
                        'actions_24h',
                        'recommendations'
                    ]
                }
            )
        
        print("✓ Narrative synthesis complete!\n")
        
        # Display executive briefing
        print("=" * 80)
        print("EXECUTIVE BRIEFING")
        print("=" * 80)
        print(f"\nTimeline start: {source_rows[0]['event_time']}")
        print(f"Timeline end:   {source_rows[-1]['event_time']}")
        print(f"Event count:    {len(source_rows)}")
        
# Risk score (if endpoint provides it)
        risk_score = narrative_result.get('risk_score', 0)
        print(f"\nRISK SCORE: {risk_score}/100")
        if risk_score >= 80:
            print("Level: CRITICAL - Requires executive action")
        elif risk_score >= 60:
            print("Level: HIGH - Monitor closely")
        elif risk_score >= 40:
            print("Level: MEDIUM - Standard response")
        else:
            print("Level: LOW - Routine handling")
        
        # Narrative
        narrative = narrative_result.get('narrative', '')
        if narrative:
            print(f"\nINCIDENT NARRATIVE:")
            print("-" * 80)
            print(narrative)
            print("-" * 80)
        else:
            summary = narrative_result.get('explainability_summary', '')
            if summary:
                print(f"\nEXPLAINABILITY SUMMARY:")
                print("-" * 80)
                print(summary)
                print("-" * 80)
            elif 'steps' in narrative_result:
                print("\nPLANNED RESPONSE STEPS:")
                for idx, step in enumerate(narrative_result.get('steps', []), 1):
                    if isinstance(step, dict):
                        action = step.get('action', str(step))
                        tool = step.get('tool', 'n/a')
                        print(f"  {idx}. {action} (tool: {tool})")
                    else:
                        print(f"  {idx}. {step}")
        
        # Timeline analysis
        timeline_analysis = narrative_result.get('causality_chain', [])
        if timeline_analysis:
            print(f"\nCAUSALITY CHAIN ({len(timeline_analysis)} links):")
            for idx, link in enumerate(timeline_analysis, 1):
                if isinstance(link, dict):
                    cause = link.get('cause', 'Event')
                    effect = link.get('effect', 'Outcome')
                    print(f"  {idx}. {cause} -> {effect}")
                else:
                    print(f"  {idx}. {link}")
        
        # Key metrics
        metrics = narrative_result.get('key_metrics', {})
        if metrics:
            print(f"\nKEY METRICS:")
            for metric, value in metrics.items():
                print(f"  - {metric}: {value}")
        
        # 24-hour action plan
        actions_24h = narrative_result.get('actions_24h', [])
        if actions_24h:
            print(f"\n24-HOUR ACTION PLAN:")
            for idx, action in enumerate(actions_24h, 1):
                if isinstance(action, dict):
                    priority = action.get('priority', 'P2')
                    item = action.get('action', str(action))
                    print(f"  {idx}. [{priority}] {item}")
                else:
                    print(f"  {idx}. {action}")
        
        # Recommendations
        recommendations = narrative_result.get('recommendations', [])
        if recommendations:
            print(f"\nRECOMMENDATIONS:")
            for idx, rec in enumerate(recommendations, 1):
                print(f"  {idx}. {rec}")
    
    except AttributeError as e:
        print(f"ERROR: SDK method not available: {e}")
        print("Fix: Ensure SDK version 0.1.0+ is installed with explain()/plan() methods")
    except Exception as e:
        print(f"ERROR: {e}")
        print("\nTroubleshooting:")
        print("  - Backend running at https://admin.ninai.sansten.com?")
        print("  - /cognitive/gateway endpoints reachable?")
        print("  - Auth token valid?")

print("\n" + "=" * 80)
print("KEY INSIGHT")
print("=" * 80)
print("""
From raw timeline to executive story:

Input: 8 discrete timeline events with explicit datetime metadata
Output: 
  ✓ Coherent narrative connecting all events
  ✓ Risk score: quantified severity
  ✓ Causality chain: detection -> impact -> root cause -> fix -> resolution
  ✓ Key metrics: 127 affected, $120K exposure, 99.95% -> 97.8% SLA
  ✓ 24h actions: follow-up priorities (P1-P3)
  ✓ Recommendations: prevent recurrence

One API call. No manual synthesis. No reading between the lines.
""")

STEP 3: Executive Narrative Synthesis

Analyzing incident timeline with seed: 42625412

Using in-session timeline data: 8 events

INVOKING COGNITIVE GATEWAY PLAN (direct synthesis fallback)

Calling client.cognitive.gateway.plan() with timeline context...
✓ Narrative synthesis complete!

EXECUTIVE BRIEFING

RISK SCORE: 0/100
Level: LOW - Routine handling

PLANNED RESPONSE STEPS:
  1. {'step_id': 's1', 'action': 'Retrieve P1 playbook', 'tool': 'playbook.match'}
  2. {'step_id': 's2', 'action': 'Identify escalation targets', 'tool': 'org_attention'}
  3. {'step_id': 's3', 'action': 'Dispatch P1 notification', 'tool': 'action.dispatch'}
  4. {'step_id': 's4', 'action': 'Open incident episode', 'tool': 'episode.create'}

KEY INSIGHT

From raw timeline to executive story:

Input: 8 discrete timeline events (09:02 → 12:30)
Output: 
  ✓ Coherent narrative connecting all events
  ✓ Risk score: quantified severity
  ✓ Causality chain: detection → impact → root cause → fix → resolution
  ✓ Key m

## Step 4: Understanding Narrative Synthesis

### What Just Happened

You didn't write a narrative yourself. Instead:

1. **You stored timeline fragments** (8 discrete events, no structure)
2. **You called Cognitive Gateway** with `explain()` verb
3. **Ninai's backend agents synthesized understanding:**
   - NarrativeSynthesisAgent (Phase 23) — stitched events into coherent story
   - CausalReasoningAgent (Phase 12) — linked causality: spike → impact → root cause → fix
   - CredibilityAgent (Phase 19) — scored risk based on business impact metrics
   - AuditTrailAgent (Phase 31) — generated explainability chain
   - QueryIntelligenceAgent (Phase 27) — extracted key insights
4. **You received a polished executive briefing** with risk score, narrative, metrics, and action plan

### The Architecture

```
Timeline Events (fragmented)
  ├─ 09:02 Spike detected
  ├─ 09:17 Customer impact
  ├─ 09:41 Root cause found
  ├─ 10:05 Business risk
  ├─ 10:45 Fix deployed
  ├─ 11:15 Fix verified
  ├─ 12:00 Communication
  └─ 12:30 Post-incident

              (search by seed)
                    ↓
    ┌───────────────────────────────┐
    │ Cognitive Gateway: explain()  │
    ├───────────────────────────────┤
    │ • Reads timeline              │
    │ • Links causality             │
    │ • Scores risk (80/100)        │
    │ • Extracts metrics            │
    │ • Generates narrative         │
    │ • Plans 24h actions           │
    └───────────────────────────────┘
                    ↓
    Executive Brief (polished)
    ├─ Risk: CRITICAL (80/100)
    ├─ Narrative: Coherent story
    ├─ Metrics: 127 affected, $120K risk
    ├─ Actions: P1-P3 priorities
    └─ Recommendations: Prevention steps
```

### Key Insight: Fragments → Understanding

Demo 5 shows that **raw events aren't insight**. Ninai transforms fragmented timeline into:
- ✅ Coherent narrative (events make sense now)
- ✅ Risk quantification (severity is clear)
- ✅ Causality chains (why → what → how)
- ✅ Action priorities (what to do next)
- ✅ Prevention steps (how to prevent recurrence)

This is **narrative intelligence** — not just timeline storage.

### Next Steps

1. **Cognitive Gateway Mastery** — Combine all 5 verbs: write → read → decide → plan → explain
2. **Multi-Demo Workflow** — Use Demos 1-5 together in production
3. **Custom Extractors** — Define your own aggregation logic

## Step 5: Troubleshooting — If Narrative Synthesis Didn't Work

In [ ]:
# Troubleshooting: Check SDK and backend
print("=" * 80)
print("TROUBLESHOOTING: Executive Narrative Synthesis")
print("=" * 80)

# Check 1: SDK has gateway.explain
print("\n[1] Checking SDK cognitive.gateway.explain()...")
try:
    if hasattr(client, 'cognitive') and hasattr(client.cognitive.gateway, 'explain'):
        print("OK: SDK has gateway.explain() method")
        import inspect
        sig = inspect.signature(client.cognitive.gateway.explain)
        print(f"    Signature: {sig}")
    else:
        print("ERROR: SDK missing gateway.explain()")
        print("  Fix: pip install -e ./repos/ninai/sdk/python")
except Exception as e:
    print(f"ERROR: {e}")

# Check 2: Verify timeline memories exist
print("\n[2] Verifying timeline events...")
try:
    verify = client.memories.search(query=f'seed={seed}', limit=20)
    if verify.items:
        print(f"OK: Found {len(verify.items)} timeline events")
        for mem in verify.items:
            time_phase = mem.content_preview.split(' - ')[0:2]
            print(f"    - {' - '.join(time_phase)}")
    else:
        print("ERROR: No timeline events found")
        print("  Action: Go back to Step 2 and create timeline")
except Exception as e:
    print(f"ERROR: {e}")

# Check 3: Test gateway.explain() call
print("\n[3] Testing gateway.explain() method...")
try:
    test_explain = client.cognitive.gateway.explain(
        memory_id=f"test_timeline_{seed}"
    )
    print("OK: gateway.explain() endpoint is responsive")
except AttributeError as e:
    print(f"ERROR: Method not available: {e}")
    print("  Fix: Ensure SDK 0.1.0+ with CognitiveGatewayResource")
except Exception as e:
    error_str = str(e)[:60]
    print(f"ERROR: Backend endpoint missing or unreachable: {error_str}")
    print("  Action: Start backend: cd repos/ninai/backend && uvicorn app.main:app")

print("\n" + "=" * 80)
print("DEMO 5 CHECKLIST")
print("=" * 80)
print("""
Prerequisites:
  [x] SDK 0.1.0+ installed (has explain() method)
  [x] Client authenticated with valid token
  [x] Timeline events created in Step 2 (8+ events)
  [?] Backend running at https://admin.ninai.sansten.com
  [?] /cognitive/gateway/explain endpoint accessible

If Step 3 failed:
  1. Check SDK has gateway.explain() method
  2. Verify timeline events were created (Step 2)
  3. Ensure backend is running
  4. Validate auth token is valid
  5. Check /cognitive/gateway/explain endpoint exists

Expected Output Format (Executive Brief):
{
    "risk_score": 80,
    "risk_level": "CRITICAL",
    "narrative": "Coherent story of incident",
    "causality_chain": [
        {"cause": "spike", "effect": "impact"},
        {"cause": "impact", "effect": "business risk"},
        ...
    ],
    "key_metrics": {
        "affected_users": 127,
        "arr_risk": "$120K+",
        "sla_impact": "99.95% → 97.8%"
    },
    "actions_24h": [
        {"priority": "P1", "action": "..."},
        {"priority": "P2", "action": "..."}
    ],
    "recommendations": ["...", "..."]
}

Timeline fragments → Polished executive briefing!
""")